# Sprint 11 — Champion stabilization

This notebook only orchestrates package code. Run discovery, inspect and retain its report, then open confirmation exactly once from the frozen selection.

In [ ]:
from pathlib import Path

import pandas as pd

from fpl_model.evaluation import ExpandingWindowSplitter, WalkForwardHarness, load_evaluation_plan
from fpl_model.models import (
    FinalistDeclaration,
    freeze_champion_selection,
    regenerate_champion_discovery_report,
    run_champion_discovery,
    run_untouched_confirmation,
)

MODEL_FRAME = Path("data/processed/player_gameweek_model.parquet")
FINALIST_REPORT = Path("reports/09_participation_report.json")
DATA_MANIFEST_SHA256 = "<replace-with-audited-sha256>"
frame = pd.read_parquet(MODEL_FRAME)
evaluation_plan = load_evaluation_plan()
harness = WalkForwardHarness(ExpandingWindowSplitter(evaluation_plan))

In [ ]:
finalist = FinalistDeclaration.from_report(
    FINALIST_REPORT, discovery_windows=("discovery_1", "discovery_2")
)
study = run_champion_discovery(
    frame,
    harness,
    finalist,
    data_manifest_hashes={"player_gameweek_model": DATA_MANIFEST_SHA256},
)
study.write("reports/11_discovery_predictions")
discovery_report = regenerate_champion_discovery_report(study)
discovery_report.write_json("reports/11_discovery_report.json")
selection = freeze_champion_selection(
    discovery_report, study, evaluation_plan_version=evaluation_plan.version
)
selection.write_json("reports/11_selection.freeze.json")
discovery_report.selection

## Untouched confirmation

Do not edit the freeze after inspecting discovery. The immutable output path prevents accidentally replacing the first confirmation result.

In [ ]:
run_untouched_confirmation(
    frame,
    harness,
    selection,
    artifact_path="reports/11_confirmation.parquet",
)